# Pipeline Walk Forward · Triple Barrier · Greedy Feature Selection · EBM

## Arquitectura de la pipeline

```
Precio / Features brutas
        │
        ▼
┌─────────────────────┐
│  Triple Barrier     │  → Etiquetas {-1, 0, +1}
│  Labeling           │    -1: stop loss tocado primero
└─────────┬───────────┘    +1: take profit tocado primero
          │                 0: barrera vertical (tiempo)
          ▼
┌─────────────────────┐
│  Walk Forward       │  → Folds temporales con zona de
│  + Embargo          │    embargo entre train y test
└─────────┬───────────┘    (evita fuga de información)
          │
          ▼
┌─────────────────────┐
│  Greedy Forward     │  → Selecciona features una a una
│  Feature Selection  │    evaluando AUC con EBM en CV
└─────────┬───────────┘
          │
          ▼
┌─────────────────────┐
│  EBM Classifier     │  → Explainable Boosting Machine
│  (InterpretML)      │    Entrenamiento final por fold
└─────────┬───────────┘
          │
          ▼
   Métricas + PnL walk-forward + Feature Importance
```

**Referencia principal:** López de Prado, M. (2018). *Advances in Financial Machine Learning*. Wiley.

In [ ]:
# !pip install interpret scikit-learn pandas numpy matplotlib seaborn

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from typing import Iterator, List, Tuple, Optional

from sklearn.base import clone
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.preprocessing import label_binarize

from interpret.glassbox import ExplainableBoostingClassifier

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})
np.random.seed(42)

print('Librerías cargadas.')

## 1. Datos

Sustituye `generate_synthetic_ohlcv()` por tu propio DataFrame con columnas `open`, `high`, `low`, `close`, `volume`.  
El índice debe ser `DatetimeIndex` con frecuencia diaria.

In [ ]:
def generate_synthetic_ohlcv(n: int = 2000, start: str = '2015-01-01') -> pd.DataFrame:
    """Genera OHLCV sintético con cierta estructura (régimen + ruido)."""
    dates = pd.bdate_range(start=start, periods=n)
    # GBM con volatilidad por regímenes
    regime = np.repeat(np.random.choice([-1, 1], size=n // 100 + 1), 100)[:n]
    drift  = 0.0002 * regime
    vol    = np.where(regime == 1, 0.005, 0.010)
    log_ret = drift + vol * np.random.randn(n)
    close   = 100 * np.exp(np.cumsum(log_ret))
    noise   = lambda s: close * (1 + s * np.abs(np.random.randn(n)))
    high    = close + noise(0.003)
    low     = close - noise(0.003)
    open_   = close * np.exp(-log_ret)
    volume  = np.random.lognormal(mean=10, sigma=0.5, size=n)
    return pd.DataFrame({'open': open_, 'high': high, 'low': low,
                         'close': close, 'volume': volume}, index=dates)


# ── Cargar datos ───────────────────────────────────────────────────────────────
df_raw = generate_synthetic_ohlcv(n=2500)

# Para usar datos reales, comenta la línea anterior y descomenta:
# df_raw = pd.read_csv('tu_archivo.csv', index_col=0, parse_dates=True)

print(f'Shape: {df_raw.shape}')
print(f'Rango: {df_raw.index[0].date()} → {df_raw.index[-1].date()}')
df_raw.tail(3)

## 2. Feature Engineering

In [ ]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Construye features a partir de OHLCV.
    Amplía esta función con tus propias features (macro, sentiment, etc.).
    """
    c = df['close']
    v = df['volume']
    feats = pd.DataFrame(index=df.index)

    # ── Retornos logarítmicos ──────────────────────────────────────────────────
    for lag in [1, 2, 3, 5, 10, 21]:
        feats[f'ret_{lag}d'] = np.log(c / c.shift(lag))

    # ── Volatilidad realizada (rolling std de log-retornos) ───────────────────
    for w in [5, 10, 21, 63]:
        feats[f'vol_{w}d'] = feats['ret_1d'].rolling(w).std()

    # ── Momentum / Z-score de precio ─────────────────────────────────────────
    for w in [10, 21, 63]:
        roll_mean = c.rolling(w).mean()
        roll_std  = c.rolling(w).std()
        feats[f'zscore_{w}d'] = (c - roll_mean) / (roll_std + 1e-8)

    # ── Ratio de volatilidad (régimen) ────────────────────────────────────────
    feats['vol_ratio_5_21']  = feats['vol_5d']  / (feats['vol_21d']  + 1e-8)
    feats['vol_ratio_21_63'] = feats['vol_21d'] / (feats['vol_63d']  + 1e-8)

    # ── ATR normalizado ───────────────────────────────────────────────────────
    tr = pd.concat([
        df['high'] - df['low'],
        (df['high'] - c.shift(1)).abs(),
        (df['low']  - c.shift(1)).abs()
    ], axis=1).max(axis=1)
    feats['atr_14d'] = tr.rolling(14).mean() / (c + 1e-8)

    # ── Volumen relativo ──────────────────────────────────────────────────────
    feats['vol_rel_21d'] = v / (v.rolling(21).mean() + 1e-8)

    # ── Distancia a máx/mín rolling ──────────────────────────────────────────
    feats['dist_high_21d'] = c / (df['high'].rolling(21).max() + 1e-8) - 1
    feats['dist_low_21d']  = c / (df['low'].rolling(21).min()  + 1e-8) - 1

    return feats.dropna()


features = build_features(df_raw)
print(f'Features: {features.shape[1]} columnas, {features.shape[0]} observaciones')
print(list(features.columns))

## 3. Triple Barrier Labeling

Dado un evento en $t$, se definen tres barreras:

| Barrera | Definición | Señal |
|---|---|---|
| **Superior** (take profit) | $p_t \cdot (1 + \text{pt} \cdot \sigma_t)$ | +1 |
| **Inferior** (stop loss) | $p_t \cdot (1 - \text{sl} \cdot \sigma_t)$ | −1 |
| **Vertical** (tiempo) | $t + \text{holding\_days}$ | signo del retorno |

La etiqueta es la de la barrera tocada primero.

In [ ]:
def ewma_volatility(close: pd.Series, span: int = 20) -> pd.Series:
    """Volatilidad diaria EWMA sobre log-retornos."""
    log_ret = np.log(close / close.shift(1))
    return log_ret.ewm(span=span, min_periods=span).std()


def triple_barrier_labels(
    close: pd.Series,
    t_events: pd.DatetimeIndex,
    pt: float = 1.5,
    sl: float = 1.5,
    holding_days: int = 10,
    vol_span: int = 20,
    vertical_label: str = 'sign',   # 'sign' | 'zero'
) -> pd.DataFrame:
    """
    Etiquetado Triple Barrera (López de Prado, cap. 3).

    Parámetros
    ----------
    close        : serie de precios de cierre
    t_events     : fechas de entrada (eventos)
    pt           : multiplicador barrera superior  (take profit)
    sl           : multiplicador barrera inferior  (stop loss)
    holding_days : días máximos de la barrera vertical
    vol_span     : ventana EWMA para la volatilidad
    vertical_label: qué etiqueta dar cuando toca la barrera vertical:
                    'sign' → signo del retorno, 'zero' → siempre 0

    Retorna
    -------
    DataFrame con columnas: t1 (fecha de salida), ret, label
    """
    vol   = ewma_volatility(close, span=vol_span).reindex(t_events)
    out   = []

    for t0 in t_events:
        p0 = close.loc[t0]
        # Barrera vertical
        future = close.loc[t0:]
        if len(future) < 2:
            continue
        idx_end = future.index.searchsorted(t0 + pd.offsets.BDay(holding_days))
        idx_end = min(idx_end, len(future) - 1)
        t1 = future.index[idx_end]
        window = future.iloc[1: idx_end + 1]

        if window.empty:
            continue

        sigma = vol.loc[t0] if not np.isnan(vol.loc[t0]) else 0.0

        upper = p0 * (1 + pt * sigma)
        lower = p0 * (1 - sl * sigma)

        # Primera vez que se tocan las barreras horizontales
        hit_up   = window[window >= upper]
        hit_down = window[window <= lower]

        t_up   = hit_up.index[0]   if not hit_up.empty   else pd.NaT
        t_down = hit_down.index[0] if not hit_down.empty else pd.NaT

        # Determinar qué barrera se tocó primero
        if pd.isna(t_up) and pd.isna(t_down):
            # Solo barrera vertical
            exit_date = t1
            ret = np.log(close.loc[t1] / p0)
            label = int(np.sign(ret)) if vertical_label == 'sign' else 0
        elif pd.isna(t_down) or (not pd.isna(t_up) and t_up <= t_down):
            exit_date = t_up
            ret = np.log(close.loc[t_up] / p0)
            label = 1
        else:
            exit_date = t_down
            ret = np.log(close.loc[t_down] / p0)
            label = -1

        out.append({'t0': t0, 't1': exit_date, 'ret': ret, 'label': label})

    return pd.DataFrame(out).set_index('t0')


# ── Configuración ─────────────────────────────────────────────────────────────
TB_PARAMS = dict(
    pt           = 1.5,    # take profit: 1.5× σ diario
    sl           = 1.5,    # stop loss:   1.5× σ diario
    holding_days = 10,     # barrera vertical: 10 días hábiles
    vol_span     = 20,     # EWMA 20 días para la volatilidad
    vertical_label = 'sign',
)

# Eventos: usamos todos los días hábiles alineados con features
close_aligned = df_raw['close'].reindex(features.index)
t_events      = features.index

labels = triple_barrier_labels(close_aligned, t_events, **TB_PARAMS)
print(f'Etiquetas generadas: {len(labels)}')
print(labels['label'].value_counts().sort_index())

In [ ]:
# ── Visualización: distribución de etiquetas y retornos ──────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Precio con colores por etiqueta
colors = {-1: '#e74c3c', 0: '#95a5a6', 1: '#27ae60'}
axes[0].plot(close_aligned.index, close_aligned.values, lw=0.6, color='#2c6fad', zorder=1)
for lbl, grp in labels.groupby('label'):
    axes[0].scatter(grp.index, close_aligned.reindex(grp.index),
                    s=4, color=colors[lbl], zorder=2, label=f'label={lbl:+d}')
axes[0].set_title('Precio con etiquetas Triple Barrier', fontweight='bold')
axes[0].set_ylabel('Precio')
axes[0].legend(fontsize=8, markerscale=3)

# Distribución de etiquetas
cnt = labels['label'].value_counts().sort_index()
bars = axes[1].bar([str(k) for k in cnt.index], cnt.values,
                   color=[colors[k] for k in cnt.index], edgecolor='white')
for bar, v in zip(bars, cnt.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 f'{v}\n({v/len(labels)*100:.1f}%)', ha='center', fontsize=9)
axes[1].set_title('Distribución de etiquetas', fontweight='bold')
axes[1].set_xlabel('Etiqueta')
axes[1].set_ylabel('Frecuencia')

# Distribución de retornos por etiqueta
for lbl in [-1, 0, 1]:
    subset = labels[labels['label'] == lbl]['ret']
    axes[2].hist(subset, bins=40, alpha=0.6, color=colors[lbl], label=f'{lbl:+d}',
                 edgecolor='white', density=True)
axes[2].set_title('Distribución de retornos por etiqueta', fontweight='bold')
axes[2].set_xlabel('Log-retorno hasta salida')
axes[2].legend()

plt.suptitle('Triple Barrier Labeling', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Walk Forward con Embargo

El **embargo** añade una zona muerta entre el final del train y el inicio del test para evitar que observaciones con solapamiento de la barrera vertical contaminen la evaluación.

```
│◄────── Train ───────►│◄─Embargo─►│◄────── Test ──────►│
                        └── int(embargo_pct × n_test) ──┘
```

In [ ]:
class WalkForwardEmbargo:
    """
    Walk-forward cross-validator con zona de embargo.

    Compatible con la interfaz sklearn (split / get_n_splits).

    Parámetros
    ----------
    n_splits    : número de folds de test
    test_size   : fracción del total para cada test (0 < test_size < 1)
    embargo_pct : fracción del tamaño de test que se embarga tras el train
    """

    def __init__(self, n_splits: int = 5, test_size: float = 0.10,
                 embargo_pct: float = 0.01):
        self.n_splits    = n_splits
        self.test_size   = test_size
        self.embargo_pct = embargo_pct

    def split(self, X, y=None, groups=None) -> Iterator[Tuple[np.ndarray, np.ndarray]]:
        n = len(X)
        n_test    = max(1, int(n * self.test_size))
        n_embargo = max(0, int(n_test * self.embargo_pct))

        # Primer inicio del test: dejamos suficiente historia para entrenar
        step = n_test
        starts = [n - (self.n_splits - k) * step for k in range(self.n_splits)]

        for test_start in starts:
            test_end   = min(test_start + n_test, n)
            train_end  = test_start - n_embargo   # zona de embargo antes del test
            if train_end <= 0:
                continue
            train_idx = np.arange(0, train_end)
            test_idx  = np.arange(test_start, test_end)
            yield train_idx, test_idx

    def get_n_splits(self, X=None, y=None, groups=None) -> int:
        return self.n_splits


# ── Parámetros del walk-forward ───────────────────────────────────────────────
WF_PARAMS = dict(
    n_splits    = 5,
    test_size   = 0.12,   # ~12% del total en cada test
    embargo_pct = 0.02,   # 2% del test como embargo
)
wf_cv = WalkForwardEmbargo(**WF_PARAMS)


# ── Alineación: X, y, ret deben tener el mismo índice ────────────────────────
common_idx = features.index.intersection(labels.index)
X   = features.loc[common_idx]
y   = labels.loc[common_idx, 'label']
ret = labels.loc[common_idx, 'ret']

print(f'Dataset final: {X.shape[0]} obs × {X.shape[1]} features')
print(f'Clases: {sorted(y.unique())}')

In [ ]:
# ── Visualización de los folds walk-forward ───────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 3.5))
n = len(X)
n_test    = int(n * WF_PARAMS['test_size'])
n_embargo = int(n_test * WF_PARAMS['embargo_pct'])

for fold, (tr_idx, te_idx) in enumerate(wf_cv.split(X)):
    y_pos = fold
    # Train
    ax.barh(y_pos, len(tr_idx), left=tr_idx[0], height=0.5,
            color='#3498db', alpha=0.7)
    # Embargo
    emb_start = tr_idx[-1] + 1
    ax.barh(y_pos, n_embargo, left=emb_start, height=0.5,
            color='#e67e22', alpha=0.9)
    # Test
    ax.barh(y_pos, len(te_idx), left=te_idx[0], height=0.5,
            color='#27ae60', alpha=0.7)
    ax.text(te_idx[-1] + 5, y_pos, f'Fold {fold+1}', va='center', fontsize=9)

legend_elements = [
    mpatches.Patch(color='#3498db', alpha=0.7, label='Train'),
    mpatches.Patch(color='#e67e22', alpha=0.9, label='Embargo'),
    mpatches.Patch(color='#27ae60', alpha=0.7, label='Test'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
ax.set_xlabel('Índice de observación')
ax.set_yticks([])
ax.set_title('Walk Forward con Embargo – Estructura de los Folds', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Greedy Forward Feature Selection con EBM

**Algoritmo:**
1. Empezar con conjunto vacío de features seleccionadas.
2. Para cada feature candidata: añadirla al conjunto actual y evaluar AUC (media sobre los folds del walk-forward).
3. Quedarse con la feature que maximiza el AUC.
4. Repetir hasta que ninguna feature mejore el umbral mínimo (`min_improvement`).

> La selección se hace **una sola vez** sobre todos los folds para determinar el subconjunto óptimo. El EBM final se entrena fold a fold con las features ya seleccionadas.

In [ ]:
def score_features(
    X: pd.DataFrame,
    y: pd.Series,
    feature_set: List[str],
    cv: WalkForwardEmbargo,
    model_params: dict,
) -> float:
    """
    Evalúa un subconjunto de features con EBM en walk-forward.
    Devuelve AUC OvR promediado sobre los folds.
    """
    classes = sorted(y.unique())
    aucs = []
    for tr_idx, te_idx in cv.split(X):
        X_tr, y_tr = X.iloc[tr_idx][feature_set], y.iloc[tr_idx]
        X_te, y_te = X.iloc[te_idx][feature_set], y.iloc[te_idx]
        if len(y_tr.unique()) < 2:
            continue
        ebm = ExplainableBoostingClassifier(**model_params)
        ebm.fit(X_tr, y_tr)
        proba = ebm.predict_proba(X_te)
        try:
            auc = roc_auc_score(y_te, proba, multi_class='ovr',
                                labels=classes, average='macro')
        except ValueError:
            auc = 0.5
        aucs.append(auc)
    return float(np.mean(aucs)) if aucs else 0.5


def greedy_forward_selection(
    X: pd.DataFrame,
    y: pd.Series,
    cv: WalkForwardEmbargo,
    model_params: dict,
    max_features: Optional[int] = None,
    min_improvement: float = 0.001,
    verbose: bool = True,
) -> Tuple[List[str], pd.DataFrame]:
    """
    Selección greedy hacia adelante, una feature cada vez.

    Devuelve
    --------
    selected  : lista de features seleccionadas en orden
    history   : DataFrame con la evolución del AUC por paso
    """
    all_features = list(X.columns)
    selected: List[str] = []
    remaining = all_features.copy()
    best_score = 0.5
    history = []

    if max_features is None:
        max_features = len(all_features)

    if verbose:
        print(f'{"Paso":<5} {"Feature añadida":<25} {"AUC":<10} {"Δ AUC":<10}')
        print('─' * 52)

    for step in range(max_features):
        scores_this_step = {}
        for feat in remaining:
            candidate = selected + [feat]
            sc = score_features(X, y, candidate, cv, model_params)
            scores_this_step[feat] = sc

        best_feat  = max(scores_this_step, key=scores_this_step.get)
        best_new   = scores_this_step[best_feat]
        delta      = best_new - best_score

        history.append({
            'step': step + 1,
            'feature': best_feat,
            'auc': best_new,
            'delta': delta,
        })

        if delta < min_improvement:
            if verbose:
                print(f'  → Parada: mejora {delta:.5f} < umbral {min_improvement}')
            break

        selected.append(best_feat)
        remaining.remove(best_feat)
        best_score = best_new

        if verbose:
            print(f'{step+1:<5} {best_feat:<25} {best_new:.5f}    {delta:+.5f}')

    return selected, pd.DataFrame(history)


# ── Parámetros del EBM ────────────────────────────────────────────────────────
EBM_PARAMS = dict(
    max_bins         = 256,
    max_interaction_bins = 32,
    interactions     = 5,
    learning_rate    = 0.01,
    min_samples_leaf = 2,
    random_state     = 42,
)

# ── Ejecutar selección ────────────────────────────────────────────────────────
print('Iniciando greedy forward selection...\n')
selected_features, sel_history = greedy_forward_selection(
    X, y,
    cv           = wf_cv,
    model_params = EBM_PARAMS,
    max_features = 12,
    min_improvement = 0.001,
    verbose      = True,
)
print(f'\nFeatures seleccionadas ({len(selected_features)}): {selected_features}')

In [ ]:
# ── Evolución del AUC durante la selección ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(sel_history['step'], sel_history['auc'],
             marker='o', color='#2c6fad', lw=2, ms=7)
axes[0].axhline(0.5, color='gray', ls='--', lw=1, label='Random (0.50)')
for _, row in sel_history.iterrows():
    axes[0].annotate(row['feature'], (row['step'], row['auc']),
                     textcoords='offset points', xytext=(0, 8),
                     fontsize=7, ha='center', rotation=30)
axes[0].set_xlabel('Paso (nº de features)')
axes[0].set_ylabel('AUC macro OvR (walk-forward)')
axes[0].set_title('Evolución del AUC – Greedy Forward Selection', fontweight='bold')
axes[0].legend()

axes[1].bar(sel_history['feature'], sel_history['delta'],
            color=['#27ae60' if d > 0 else '#e74c3c' for d in sel_history['delta']],
            edgecolor='white')
axes[1].axhline(0, color='black', lw=0.8)
axes[1].axhline(0.001, color='orange', ls='--', lw=1, label='umbral mínimo')
axes[1].set_xlabel('Feature añadida')
axes[1].set_ylabel('Δ AUC')
axes[1].set_title('Mejora marginal por feature', fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Entrenamiento Walk-Forward Final con EBM

Con las features seleccionadas, entrenamos el EBM en cada fold y evaluamos out-of-sample. Guardamos predicciones, probabilidades y retornos para el análisis posterior.

In [ ]:
X_sel = X[selected_features]
classes = sorted(y.unique())

fold_results = []
trained_models = []

for fold, (tr_idx, te_idx) in enumerate(wf_cv.split(X_sel)):
    X_tr, y_tr = X_sel.iloc[tr_idx], y.iloc[tr_idx]
    X_te, y_te = X_sel.iloc[te_idx], y.iloc[te_idx]
    ret_te     = ret.iloc[te_idx]

    ebm = ExplainableBoostingClassifier(**EBM_PARAMS)
    ebm.fit(X_tr, y_tr)
    trained_models.append(ebm)

    y_pred  = ebm.predict(X_te)
    y_proba = ebm.predict_proba(X_te)

    try:
        auc = roc_auc_score(y_te, y_proba, multi_class='ovr',
                            labels=classes, average='macro')
    except ValueError:
        auc = np.nan

    acc = (y_pred == y_te).mean()

    # PnL: si predice +1 → long, -1 → short, 0 → flat
    pnl = pd.Series(y_pred, index=X_te.index, dtype=float) * ret_te
    cum_pnl = pnl.cumsum()

    fold_results.append({
        'fold'      : fold + 1,
        'train_start': X_sel.index[tr_idx[0]].date(),
        'train_end'  : X_sel.index[tr_idx[-1]].date(),
        'test_start' : X_sel.index[te_idx[0]].date(),
        'test_end'   : X_sel.index[te_idx[-1]].date(),
        'n_train'   : len(tr_idx),
        'n_test'    : len(te_idx),
        'auc'       : auc,
        'accuracy'  : acc,
        'cum_pnl'   : cum_pnl.iloc[-1],
        'pnl_series': pnl,
        'y_true'    : y_te,
        'y_pred'    : pd.Series(y_pred, index=X_te.index),
    })

    print(f'Fold {fold+1} | train: {len(tr_idx):>4} | test: {len(te_idx):>3} '
          f'| AUC: {auc:.4f} | Acc: {acc:.3f} | PnL: {cum_pnl.iloc[-1]:+.4f}')

summary = pd.DataFrame([{k: v for k, v in r.items()
                          if k not in ('pnl_series','y_true','y_pred')}
                         for r in fold_results])
print('\n', summary.to_string(index=False))

## 7. Resultados y Visualización

In [ ]:
# ── PnL walk-forward acumulado ───────────────────────────────────────────────
all_pnl = pd.concat([r['pnl_series'] for r in fold_results]).sort_index()
cum_pnl_total = all_pnl.cumsum()

# Benchmark: buy-and-hold (siempre long, label = signo del retorno del activo)
bh_ret   = ret.reindex(all_pnl.index)
cum_bh   = bh_ret.cumsum()

fig = plt.figure(figsize=(14, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

# Panel 1: PnL acumulado
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(cum_pnl_total.index, cum_pnl_total.values,
         lw=1.8, color='#2c6fad', label='EBM Walk-Forward')
ax1.plot(cum_bh.index, cum_bh.values,
         lw=1.2, color='#95a5a6', ls='--', label='Buy & Hold')
# Separadores de fold
for r in fold_results:
    ax1.axvline(pd.Timestamp(r['test_start']), color='orange',
                lw=0.7, ls=':', alpha=0.8)
ax1.axhline(0, color='black', lw=0.6)
ax1.fill_between(cum_pnl_total.index, cum_pnl_total.values, 0,
                 where=cum_pnl_total.values >= 0, color='#27ae60', alpha=0.12)
ax1.fill_between(cum_pnl_total.index, cum_pnl_total.values, 0,
                 where=cum_pnl_total.values < 0, color='#e74c3c', alpha=0.12)
ax1.set_title('PnL Walk-Forward Acumulado (log-retorno)', fontweight='bold', fontsize=12)
ax1.set_ylabel('PnL acumulado')
ax1.legend(fontsize=9)

# Panel 2: AUC por fold
ax2 = fig.add_subplot(gs[1, 0])
folds_  = [r['fold'] for r in fold_results]
aucs_   = [r['auc'] for r in fold_results]
bars = ax2.bar(folds_, aucs_, color='#3498db', edgecolor='white', alpha=0.8)
ax2.axhline(0.5, color='gray', ls='--', lw=1, label='Random')
ax2.axhline(np.nanmean(aucs_), color='#c0392b', ls='--', lw=1.5,
            label=f'Media: {np.nanmean(aucs_):.4f}')
for bar, v in zip(bars, aucs_):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
             f'{v:.4f}', ha='center', fontsize=8)
ax2.set_xlabel('Fold')
ax2.set_ylabel('AUC macro OvR')
ax2.set_title('AUC por Fold', fontweight='bold')
ax2.legend(fontsize=8)
ax2.set_ylim(0.4, min(1.0, max(aucs_) + 0.05))

# Panel 3: Matriz de confusión (todos los folds)
ax3 = fig.add_subplot(gs[1, 1])
all_true = pd.concat([r['y_true'] for r in fold_results])
all_pred = pd.concat([r['y_pred'] for r in fold_results])
cm = confusion_matrix(all_true, all_pred, labels=classes)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues',
            xticklabels=classes, yticklabels=classes,
            ax=ax3, cbar_kws={'label': '%'})
ax3.set_xlabel('Predicho')
ax3.set_ylabel('Real')
ax3.set_title('Matriz de Confusión (% por fila, todos los folds)', fontweight='bold')

plt.suptitle('Pipeline Walk-Forward EBM – Resultados', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Feature importance del último modelo entrenado ───────────────────────────
last_ebm = trained_models[-1]
ebm_explain = last_ebm.explain_global(name='EBM Feature Importance')

importances = pd.Series(
    ebm_explain.data()['scores'],
    index=ebm_explain.data()['names']
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, max(4, len(importances) * 0.4)))
colors_imp = ['#27ae60' if v >= 0 else '#e74c3c' for v in importances.values]
ax.barh(importances.index, importances.values, color=colors_imp, edgecolor='white')
ax.axvline(0, color='black', lw=0.8)
ax.set_title('EBM Feature Importance – Último fold', fontweight='bold')
ax.set_xlabel('Importancia media absoluta')
plt.tight_layout()
plt.show()

print('\nClassification report (todos los folds, out-of-sample):')
print(classification_report(all_true, all_pred,
                             target_names=[f'label={c}' for c in classes]))

In [ ]:
# ── Métricas de rendimiento walk-forward ─────────────────────────────────────
total_ret   = float(cum_pnl_total.iloc[-1])
n_days      = len(all_pnl)
ann_ret     = total_ret * (252 / n_days)
ann_vol     = all_pnl.std() * np.sqrt(252)
sharpe      = ann_ret / (ann_vol + 1e-8)

rolling_max = cum_pnl_total.cummax()
drawdown    = cum_pnl_total - rolling_max
max_dd      = drawdown.min()
calmar      = ann_ret / (abs(max_dd) + 1e-8)

hit_rate    = (all_pnl > 0).sum() / (all_pnl != 0).sum()

print('══════════════════════════════════════════')
print('  MÉTRICAS WALK-FORWARD (out-of-sample)  ')
print('══════════════════════════════════════════')
print(f'  Retorno total (log)   : {total_ret:+.4f}')
print(f'  Retorno anualizado    : {ann_ret:+.4f}')
print(f'  Volatilidad anual.    :  {ann_vol:.4f}')
print(f'  Sharpe ratio          :  {sharpe:+.3f}')
print(f'  Max Drawdown          :  {max_dd:.4f}')
print(f'  Calmar ratio          :  {calmar:.3f}')
print(f'  Hit rate              :  {hit_rate:.3f}')
print(f'  AUC medio             :  {np.nanmean(aucs_):.4f}')
print(f'  Features seleccionadas:  {len(selected_features)}')
print('══════════════════════════════════════════')

## 8. Resumen y Notas

| Componente | Parámetros clave | Dónde ajustar |
|---|---|---|
| **Triple Barrier** | `pt`, `sl`, `holding_days`, `vol_span` | `TB_PARAMS` |
| **Walk Forward** | `n_splits`, `test_size`, `embargo_pct` | `WF_PARAMS` |
| **Greedy Selection** | `max_features`, `min_improvement` | llamada a `greedy_forward_selection()` |
| **EBM** | `max_bins`, `interactions`, `learning_rate` | `EBM_PARAMS` |

### Extensiones recomendadas
- **Purging**: además del embargo, eliminar del train las observaciones cuyo `t1` (salida) caiga dentro del periodo de test.
- **Sample weights**: ponderar las observaciones por el valor absoluto del retorno (`log|ret|`) para dar más peso a las señales fuertes.
- **Barreras asimétricas**: usar `pt ≠ sl` para reflejar asimetría en la distribución de retornos.
- **Meta-labeling** (López de Prado cap. 4): entrenar un segundo modelo que prediga *si el modelo primario acierta*, para filtrar señales de baja confianza.